# Advent of Code 2015 - Day 14

## Imports

In [ ]:
import marimo as mo
import plotly.graph_objects as go
import plotly.express as px

from aoc.utils import read_data

## Day 14: Reindeer Olympics

### Part 1 - Instructions

This year is the Reindeer Olympics! Reindeer can fly at high speeds, but must rest occasionally to recover their energy. Santa would like to know which of his reindeer is fastest, and so he has them race.

Reindeer can only either be flying (always at their top speed) or resting (not moving at all), and always spend whole seconds in either state.

For example, suppose you have the following Reindeer:

- Comet can fly 14 km/s for 10 seconds, but then must rest for 127 seconds.
- Dancer can fly 16 km/s for 11 seconds, but then must rest for 162 seconds.

After one second, Comet has gone 14 km, while Dancer has gone 16 km. After ten seconds, Comet has gone 140 km, while Dancer has gone 160 km. On the eleventh second, Comet begins resting (staying at 140 km), and Dancer continues on for a total distance of 176 km. On the 12th second, both reindeer are resting. They continue to rest until the 138th second, when Comet flies for another ten seconds. On the 174th second, Dancer flies for another 11 seconds.

In this example, after the 1000th second, both reindeer are resting, and Comet is in the lead at 1120 km (poor Dancer has only gotten 1056 km by that point). So, in this situation, Comet would win (if the race ended at 1000 seconds).

Given the descriptions of each reindeer (in your puzzle input), after exactly 2503 seconds, what distance has the winning reindeer traveled?

### Part 1 - Solution

In [ ]:
def exercise_14_1_find_fastest_reinder(
    reindeer_descriptions: list[str], total_time: int = 2503
) -> tuple[dict, int]:
    """
    Simulate each reindeer's flight and rest cycles to find the winning distance.

    1. Parse each reindeer description into its speed, move duration, rest
       duration, and combined cycle duration.
    2. Step through every second up to total_time, tracking each reindeer's
       cumulative distance based on whether it is flying or resting.
    3. Return the distance traveled by the reindeer that is furthest ahead.

    Args:
        reindeer_descriptions (list[str]): Raw puzzle input lines, one per
            reindeer.
        total_time (int): Number of seconds the race runs for.

    Returns:
        reindeers (dict[str, dict[str, Any]]): Mapping of reindeer name to
            its stats, including the full "traveled" history of
            [time, distance] pairs.
        winning_reindeer_distance_traveled (int): Distance traveled by the
            leading reindeer once the race ends.
    """
    reindeers = {}

    for reindeer_description in reindeer_descriptions:
        reindeer_data = reindeer_description.split()

        reindeers[reindeer_data[0]] = {
            "move_speed": int(reindeer_data[3]),
            "move_duration": int(reindeer_data[6]),
            "rest_speed": 0,
            "rest_duration": int(reindeer_data[13]),
            "cycle_duration": int(reindeer_data[6]) + int(reindeer_data[13]),
            "traveled": [[0, 0]],
        }

    for t in range(total_time + 1):
        for reindeer_name, reindeer_stats in reindeers.items():
            t_cycle = t % reindeer_stats["cycle_duration"]

            if t_cycle > 0 and t_cycle <= reindeer_stats["move_duration"]:
                reindeers[reindeer_name]["traveled"].append(
                    [
                        t,
                        reindeer_stats["traveled"][-1][1]
                        + reindeer_stats["move_speed"],
                    ]
                )
            else:
                reindeers[reindeer_name]["traveled"].append(
                    [
                        t,
                        reindeer_stats["traveled"][-1][1]
                        + reindeer_stats["rest_speed"],
                    ]
                )

    winning_reindeer_distance_traveled = max(
        [reindeer_stats["traveled"][-1][1] for _, reindeer_stats in reindeers.items()]
    )

    return reindeers, winning_reindeer_distance_traveled

In [ ]:
def exercise_14_plot_reindeer_distances(reindeers: dict) -> go.Figure:  # pyright: ignore[reportInvalidTypeForm]
    """
    Plot each reindeer's distance (and points, if tracked) over time.

    1. Check whether any reindeer has a "points" history, since only part 2
       tracks it.
    2. Create an empty figure.
    3. For each reindeer, unpack its "traveled" (and "points") history into
       time/value sequences and add each as a line trace, using a solid
       line for distance and a dashed line for points, sharing color per
       reindeer.
    4. Apply layout styling (titles, hover mode) and return the figure.

    Args:
        reindeers (dict[str, dict[str, Any]]): Mapping of reindeer name to
            its stats, including a "traveled" list of [time, distance]
            pairs and, for part 2, a "points" list of [time, points] pairs.

    Returns:
        figure (go.Figure): Interactive Plotly figure with one distance
            line per reindeer, plus a points line per reindeer when
            tracked, sharing colors and a y-axis across both metrics.
    """
    has_points = any(
        "points" in reindeer_stats for reindeer_stats in reindeers.values()
    )

    figure = go.Figure()
    colors = px.colors.qualitative.Plotly

    for index, (reindeer_name, reindeer_stats) in enumerate(reindeers.items()):
        color = colors[index % len(colors)]
        times, distances = zip(*reindeer_stats["traveled"])

        figure.add_trace(
            go.Scatter(
                x=times,
                y=distances,
                mode="lines",
                name=f"{reindeer_name} (distance)",
                legendgroup=reindeer_name,
                line={"color": color, "dash": "solid"},
                hovertemplate=(
                    f"{reindeer_name}<br>"
                    "Time: %{x}s<br>"
                    "Distance: %{y}km<extra></extra>"
                ),
            )
        )

        if has_points:
            point_times, points = zip(*reindeer_stats["points"])

            figure.add_trace(
                go.Scatter(
                    x=point_times,
                    y=points,
                    mode="lines",
                    name=f"{reindeer_name} (points)",
                    legendgroup=reindeer_name,
                    line={"color": color, "dash": "dot"},
                    hovertemplate=("Points: %{y}<extra></extra>"),
                )
            )

    figure.update_layout(
        title="Reindeer Race: Distance and Points Over Time",
        xaxis_title="Time (seconds)",
        yaxis_title="Distance (km) / Points",
        hovermode="x unified",
        template="plotly_white",
    )

    return figure

In [ ]:
reindeer_descriptions: list[str] = read_data(
    file_path="data/2015/2015_day_14.txt", separator="\n"
)

reindeers_14_1, winning_reindeer_distance_traveled = (
    exercise_14_1_find_fastest_reinder(reindeer_descriptions=reindeer_descriptions)
)

assert winning_reindeer_distance_traveled == 2660

In [ ]:
exercise_14_plot_reindeer_distances(reindeers=reindeers_14_1)

<marimo-plotly data-figure='{"data":[{"hovertemplate":"Vixen\u003cbr\u003eTime: %{x}s\u003cbr\u003eDistance: %{y}km\u003cextra\u003e\u003c/extra\u003e","legendgroup":"Vixen","line":{"color":"#636EFA","dash":"solid"},"mode":"lines","name":"Vixen (distance)","x":[0,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,96

### Part 2 - Instructions

Seeing how reindeer move in bursts, Santa decides he's not pleased with the old scoring system.

Instead, at the end of each second, he awards one point to the reindeer currently in the lead. (If there are multiple reindeer tied for the lead, they each get one point.) He keeps the traditional 2503 second time limit, of course, as doing otherwise would be entirely ridiculous.

Given the example reindeer from above, after the first second, Dancer is in the lead and gets one point. He stays in the lead until several seconds into Comet's second burst: after the 140th second, Comet pulls into the lead and gets his first point. Of course, since Dancer had been in the lead for the 139 seconds before that, he has accumulated 139 points by the 140th second.

After the 1000th second, Dancer has accumulated 689 points, while poor Comet, our old champion, only has 312. So, with the new scoring system, Dancer would win (if the race ended at 1000 seconds).

Again given the descriptions of each reindeer (in your puzzle input), after exactly 2503 seconds, how many points does the winning reindeer have?

### Part 2 - Solution

In [ ]:
def exercise_14_2_find_fastest_reindeer(
    reindeer_descriptions: list[str], total_time: int = 2503
) -> tuple[dict, int]:
    """
    Simulate each reindeer's flight and rest cycles to find the winning score.

    1. Parse each reindeer description into its speed, move duration, rest
       duration, and combined cycle duration.
    2. Step through every second, updating each reindeer's cumulative
       distance based on whether it is flying or resting.
    3. After each second, award a point to every reindeer currently tied
       for the furthest distance, recording the running point total.
    4. Return the reindeer with the most points once the race ends.

    Args:
        reindeer_descriptions (list[str]): Raw puzzle input lines, one per
            reindeer.
        total_time (int): Number of seconds the race runs for.

    Returns:
        reindeers (dict): Mapping of reindeer name to its stats, including
            the full "traveled" and "points" histories as [time, value]
            pairs.
        winning_reindeer_most_points (int): Highest point total among all
            reindeer once the race ends.
    """
    reindeers = {}

    for reindeer_description in reindeer_descriptions:
        reindeer_data = reindeer_description.split()

        reindeers[reindeer_data[0]] = {
            "move_speed": int(reindeer_data[3]),
            "move_duration": int(reindeer_data[6]),
            "rest_speed": 0,
            "rest_duration": int(reindeer_data[13]),
            "cycle_duration": int(reindeer_data[6]) + int(reindeer_data[13]),
            "traveled": [[0, 0]],
            "points": [[0, 0]],
        }

    for t in range(1, total_time + 1):
        for reindeer_name, reindeer_stats in reindeers.items():
            t_cycle = t % reindeer_stats["cycle_duration"]
            current_distance = reindeer_stats["traveled"][-1][1]

            if t_cycle > 0 and t_cycle <= reindeer_stats["move_duration"]:
                reindeers[reindeer_name]["traveled"].append(
                    [t, current_distance + reindeer_stats["move_speed"]]
                )
            else:
                reindeers[reindeer_name]["traveled"].append(
                    [t, current_distance + reindeer_stats["rest_speed"]]
                )

        furthest_distance_so_far = max(
            reindeer_stats["traveled"][-1][1] for reindeer_stats in reindeers.values()
        )

        furthest_reindeer_names = [
            reindeer_name
            for reindeer_name, reindeer_stats in reindeers.items()
            if reindeer_stats["traveled"][-1][1] == furthest_distance_so_far
        ]

        for reindeer_name, reindeer_stats in reindeers.items():
            current_points = reindeer_stats["points"][-1][1]

            if reindeer_name in furthest_reindeer_names:
                reindeers[reindeer_name]["points"].append([t, current_points + 1])
            else:
                reindeers[reindeer_name]["points"].append([t, current_points])

    winning_reindeer_most_points = max(
        reindeer_stats["points"][-1][1] for reindeer_stats in reindeers.values()
    )

    return reindeers, winning_reindeer_most_points

In [ ]:
reindeers_14_2, winning_reindeer_most_points = exercise_14_2_find_fastest_reindeer(
    reindeer_descriptions=reindeer_descriptions
)

assert winning_reindeer_most_points == 1256

In [ ]:
exercise_14_plot_reindeer_distances(reindeers=reindeers_14_2)

<marimo-plotly data-figure='{"data":[{"hovertemplate":"Vixen\u003cbr\u003eTime: %{x}s\u003cbr\u003eDistance: %{y}km\u003cextra\u003e\u003c/extra\u003e","legendgroup":"Vixen","line":{"color":"#636EFA","dash":"solid"},"mode":"lines","name":"Vixen (distance)","x":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,